Welcome to the module introducing textual analysis in AccFin Research.

AccFin research nowadays is no longer limited to numerical disclosures and structual data. Textual data — such as Management Discussion & Analysis (MD&A) sections of 10-K filings, risk factor disclosures (RFD), and earnings call transcripts — provides rich information about firms’ strategies, risks, and performance, which provides many research opportunities.

This session introduces you to textual analysis in Python, with a focus on two foundational tools:

- Regular expressions (Regex): for identifying, extracting, and cleaning patterns in text, such as firm names, dates, or accounting phrases.

- `textstat` package: for computing readability metrics, enabling systematic assessment of disclosure complexity and accessibility.

You will learn how to process unstructured text, extract meaningful features, and generate empirical datasets that can be linked to financial and capital market outcomes. By the end of the course, you will have the skills to replicate influential studies in AccFin and apply textual analysis to your own projects.

**Learning Outcomes**

By the end of this session, you will be able to:

- Understand the role of textual analysis in AccFin research, particularly how narrative disclosures affect investors, regulators, and other stakeholders.

- Use Python’s `re` package to:

    - Identify patterns such as financial terms, footnotes, and forward-looking statements.

    - Clean raw disclosures by removing tables, formatting artifacts, and boilerplate text.

- Apply the `textstat` package to compute readability measures (e.g., Fog Index, Flesch Reading Ease, Dale-Chall) and interpret their implications for disclosure transparency and investor comprehension.

- Build reproducible pipelines that ingest a collection of text files (e.g., 10-K sections), extract features, and output structured datasets suitable for statistical analysis.

- Critically evaluate limitations of readability metrics and preprocessing methods, and discuss best practices for textual analysis in academic contexts.

- Integrate textual analysis with empirical AccFin research, linking disclosure characteristics to firm fundamentals, market reactions, and regulatory outcomes.


In [2]:
import pandas as pd
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

*Run the following codes for the first time you use the `nltk` package*

nltk.download('punkt')

nltk.download('cmudict')

nltk.download('punkt_tab')

### 1. Intro to Regex

Regular expressions (regex) are patterns used to match, search, extract, or replace text.

In Python, regex functionality is provided by the `re` module.

In [3]:
import re

#### 1.1. Basic Regex Concepts

Some of the commonly used patterns are as follows:

| Regex Symbol | Meaning                                      | Example                                    | 
| ------------ | -------------------------------------------- | ------------------------------------------ |
| `.`          | Any character except newline                 | `a.b` matches "acb", "a1b"                 | 
| `\d`         | Digit (0–9)                                  | `\d{4}` matches "2023"                     | 
| `\w`         | Word character (letters, digits, underscore) | `\w+` matches "Earnings2023"               |
| `\s`         | Whitespace (space, tab, newline)             | `\s+` matches spaces                       | 
| `[]`         | Character class (any inside)                 | `[A-Z]` matches capital letters            | 
| `^`          | Start of string/line                         | `^Item` matches lines starting with "Item" | 
| `$`          | End of string/line                           | `profit$` matches lines ending in "profit" | 
| `*`          | 0 or more repetitions                        | `ab*` matches "a", "ab", "abb"             | 
| `+`          | 1 or more repetitions                        | `\d+` matches "2023", "100"                | 
| `{m,n}`      | Between m and n repetitions                  | `\d{2,4}` matches "23", "2023"             | 
| `()`         | Group (capture)                              | `(20\d{2})` captures years like 2019       | 
| `\|`         | OR                                           | \`(profit \| loss)\` matches either word   |

You can refer to the Regular Expressions Cheat Sheet at https://cheatography.com/davechild/cheat-sheets/regular-expressions/.

#### 1.2. Identifying Patterns in Texts using `re.findall()`

Basic syntax
```py
list_of_str_found = re.findall(r"pattern", existing_str_var)
```

In [4]:
text = "The company reported earnings of $5.2 billion [15] and revenues of $20 billion in 2023 [16]."

# Extract monetary amounts as full strings
amounts = re.findall(r"\$\d+(?:\.\d+)?\s*(?:thousand|million|billion)", text)
print(amounts)

['$5.2 billion', '$20 billion']


In [ ]:
# Find financial terms like "earnings", "revenue", "loss"
# Use `flag=re.IGNORECASE` to make the search case-insensitive
re.findall(r"\b(earnings?|revenues?|loss(?:es)?)\b", text, flags=re.IGNORECASE)

In [ ]:
# Find footnotes look like [1], (2), or superscripts.
re.findall(r"(\[\d+\]|\(\d+\)|\^\d+)", text)

In [ ]:
# Find forward-looking statements with keywords such as "expect", "anticipate", "may"
text = "We expect revenues to increase in the future, but results may differ."

forward_looking = r"\b(expect|anticipate|believe|future|may|might|could|intend|plan|predict)\b"
re.findall(forward_looking, text, flags=re.IGNORECASE)

['expect', 'future', 'may']

In [6]:
# Similar to, but more advanced than:
Future_terms = [
    "expect", "anticipate", "believe", "future", "may",
    "might", "could", "intend", "plan", "predict"]

[term for term in Future_terms if term in text.lower()]

['expect', 'future', 'may']

#### 1.3. Cleaning Texts using `re.sub()`

Basic syntax
```py
new_str_var = re.sub(r"text to be replaced", "new content", old_str_var)
```

In [16]:
# Remove tables (such as lines full of dashes, pipes, or repeated symbols)

raw_text = """
-------------------------------------------------
| Quarter | Revenue ($m) | Profit ($m) |
-------------------------------------------------
Q1        1000           200
Forward-looking statements appear below.

Item 1: Business Overview
"""

In [17]:
cleaned1 = re.sub(r"[-|=_]{3,}", "", raw_text)
print(cleaned1)



| Quarter | Revenue ($m) | Profit ($m) |

Q1        1000           200
Forward-looking statements appear below.

Item 1: Business Overview



In [18]:
# Remove formatting artifacts (extra spaces, newlines)
cleaned2 = re.sub(r"(\||\s+)", " ", cleaned1)
cleaned2 = cleaned2.strip()
print(cleaned2)

Quarter   Revenue ($m)   Profit ($m)   Q1 1000 200 Forward-looking statements appear below. Item 1: Business Overview


In [25]:
# Remove boilerplate disclaimers
# Most 10-Ks include a “Safe Harbor” or “Forward-Looking Statements” section.

pattern = r"forward-looking statements.*?(?=item\s1)"  # until "Item 1"
cleaned3 = re.sub(pattern, "", cleaned2, flags=re.IGNORECASE | re.DOTALL)
print(cleaned3)

Quarter   Revenue ($m)   Profit ($m)   Q1 1000 200 Item 1: Business Overview


In [ ]:
# Example Project: Cleaning an MD&A section

mda = """
Item 7. Management Discussion & Analysis
----------------------------------------
Our revenues increased by 10% in 2022 [1].
We expect growth in the future, but results may differ materially.

-------------------------------------------------
| Quarter | Revenue ($m) | Profit ($m) |
-------------------------------------------------
Q1        1000           200
Q2        1100           250
Q3        1200           300
Q4        1300           350
----------------------------------------
Safe Harbor Statement: These forward-looking statements are subject to risks...

Item 7A. Quantitative and Qualitative Disclosures
"""

In [ ]:
# Remove tables
mda = re.sub(r"[-|=_]{3,}.*", "", mda)

# Remove footnotes
mda = re.sub(r"(\[\d+\]|\(\d+\))", "", mda)

# Remove boilerplate Safe Harbor
mda = re.sub(r"Safe Harbor.*?(?=Item 7A)", "", mda, flags=re.DOTALL | re.IGNORECASE)

print(mda)

### 2. Intro to `textstat`

See the offical documents at https://github.com/textstat/textstat.

In [ ]:
import textstat

In [ ]:
## 1) Sample texts

text_easy = (
    "Robots help in the warehouse. They lift boxes and move them to the right shelves. "
    "Workers use tablets to check orders. The system is simple to learn and fast to use."
)

text_medium = (
    "Management expects revenue to grow modestly as the company reallocates resources toward higher-margin segments. "
    "While this transition may reduce short-term sales, it is designed to enhance operating leverage and cash flows."
)

text_hard = (
    "The amortization of capitalized development expenditures, coupled with deferred tax adjustments arising from "
    "temporary timing differences, materially affected the period's comprehensive income and diluted earnings trajectory."
)

docs = [("Easy", text_easy), ("Medium", text_medium), ("Hard", text_hard)]

#### 2.1. Basic counts and estimates

`textstat` offers quick utilities for character, word, syllable, and sentence counts, plus estimated reading time.

In [ ]:
def basic_metrics(text: str) -> dict:
    return {
        "chars": textstat.char_count(text, ignore_spaces=True),
        "letters": textstat.letter_count(text),
        "words": textstat.lexicon_count(text, removepunct=True),
        "sentences": textstat.sentence_count(text),
        "syllables": textstat.syllable_count(text),
        "avg_sentence_length": textstat.words_per_sentence(text),
        "avg_syllables_per_word": textstat.avg_syllables_per_word(text),
    }

basic_metrics(text_medium)

#### 2.2. Calculating Readability scores (single text)

Here are the most commonly cited readability indices. Lower complexity → higher *Flesch Reading Ease*; higher grades → more difficult.

- **Gunning Fog** (`gunning_fog`)
- **SMOG Index** (`smog_index`)
- **Flesch Reading Ease** (`flesch_reading_ease`) — higher is easier (typical range ≈ 0–100).
- **Flesch–Kincaid Grade** (`flesch_kincaid_grade`)
- **Coleman–Liau** (`coleman_liau_index`)
- **Automated Readability Index** (`automated_readability_index`)
- **Dale–Chall Score** (`dale_chall_readability_score`)
- **Linsear Write** (`linsear_write_formula`)
- **Text Standard** (`text_standard`) — `textstat`’s combined “grade band” summary (string).

In [ ]:
def readability_metrics(text: str) -> dict:
    return {
        "gunning_fog": textstat.gunning_fog(text),
        "smog_index": textstat.smog_index(text),
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "coleman_liau": textstat.coleman_liau_index(text),
        "automated_readability": textstat.automated_readability_index(text),
        "dale_chall": textstat.dale_chall_readability_score(text),
        "linsear_write": textstat.linsear_write_formula(text),
        "text_standard": textstat.text_standard(text),
    }

easy = readability_metrics(text_easy)
medium = readability_metrics(text_medium)
hard = readability_metrics(text_hard)

pd.DataFrame([easy, medium, hard], index=["Easy", "Medium", "Hard"])

#### 2.3. Difficult words and vocabulary insights

- `difficult_words(text)` returns a count based on the New Dale–Chall list.
- `polysyllabcount(text)` counts words with ≥3 syllables.
- `monosyllabcount(text)` counts one-syllable words.

In [ ]:
def vocab_metrics(text: str) -> dict:
    return {
        "difficult_word_count": textstat.difficult_words(text),
        "polysyllables": textstat.polysyllabcount(text),
        "monosyllables": textstat.monosyllabcount(text),
        "lexicon_count": textstat.lexicon_count(text, removepunct=True),
        "syllables": textstat.syllable_count(text),
    }

vocab_metrics(text_hard)

### 3. Textual Analysis Resources

In [ ]:
# Get the Stop Words shared by https://sraf.nd.edu/textual-analysis/stopwords/
with open('data/StopWords_Generic.txt', 'r') as file:
	Stop_Words = file.read().splitlines()
print(len(Stop_Words))

In [ ]:
# Loughran-McDonald Master Dictionary w/ Sentiment Word Lists
# Available from https://sraf.nd.edu/loughranmcdonald-master-dictionary/
sheet_id = '1y2LVPvRqdggmIhSnHQcEZA5lYbe3vS5w'

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
LM_Dictionary = pd.read_csv(url)

In [ ]:
# List of R&D keywords from Merkley (2014 TAR)
with open('data/R&D_Keywords.txt', 'r') as file:
    RnD_Phrases = file.read().splitlines()
print(len(RnD_Phrases))

### 4. Count keywords

Downlaod the 10-X files from https://sraf.nd.edu/sec-edgar-data/. Example files saved in './data/sec_filings'.

In [ ]:
def count_RnD_sentences(text: str) -> tuple[str, int]:
    """
    Count sentences containing R&D phrases in the given text.
    """

    sentences = sent_tokenize(text) # Tokenize the text into sentences
    count = 0
    RnD_Text = ''
    
    # Check each sentence for the presence of any of the phrases
    for sentence in sentences:
        for phrase in RnD_Phrases:
            if phrase in sentence.lower():
                count += 1
                temp_sentence = re.sub(r'\s+', ' ', 
                                       sentence.replace('\n',' ').strip()
                                       )
                RnD_Text += temp_sentence + '\n'
                break  # If one phrase is found, no need to check others
    if RnD_Text != '':
        RnD_Text = RnD_Text.strip()
    return RnD_Text, count

In [ ]:
all_data = pd.DataFrame(columns = ['FileName', 'RnD_Sentences', "RnD_Text"])
root = Path('data/sec_filing')
File_List = list(root.glob('*_10-K_*.txt'))
row=0
for file_path in File_List:
    with open(file_path, 'r') as file:
        text1 = file.read()
    
    Text, Sentence_count = count_RnD_sentences(text1)

    Name = file_path.stem  # Get the file name without directory and extension

    all_data.loc[row] = (Name, Sentence_count, Text)
    row += 1

In [ ]:
print(all_data.loc[all_data['RnD_Sentences'] > 0, 'RnD_Text'])

In [ ]:
all_data['RnD_Text'] = all_data['RnD_Text'].astype(str)
all_data = all_data[all_data['RnD_Text']!='']

all_data[['FILING_DATE', 'temp']] = all_data['FileName'].str.split('_10-K_edgar_data_', expand=True)
all_data[['CIK', 'ACC_NUM']] = all_data['temp'].str.split('_', expand=True)
all_data['ACC_NUM'] = all_data['ACC_NUM'].apply(lambda x: re.sub('.txt', '', x))
all_data['CIK'] = pd.to_numeric(all_data['CIK'], errors='coerce')
all_data['FILING_DATE'] = pd.to_numeric(all_data['FILING_DATE'], errors='coerce')
all_data.dropna(subset=['CIK', 'ACC_NUM','FILING_DATE'],inplace=True)

all_data.drop(columns=['FileName', 'temp'], inplace=True)

### 5. Calculate the readability of R&D Disclosure

In [ ]:
all_data['RnD_Fog_Index'] = all_data['RnD_Text'].apply(textstat.gunning_fog)

### 6. Calculate the Sentiment of R&D Disclosure

In [ ]:
# Functions to calculate the Sentiment
Neg_Words = LM_Dictionary[LM_Dictionary['Negative']!=0]['Word'].tolist()
Pos_Words = LM_Dictionary[LM_Dictionary['Positive']!=0]['Word'].tolist()
Uncertain_Words = LM_Dictionary[LM_Dictionary['Uncertainty']!=0]['Word'].tolist()
Complex_Words = LM_Dictionary[LM_Dictionary['Complexity']!=0]['Word'].tolist()

def Pos_words(text):
    words = word_tokenize(text)
    pos_words = [word for word in words if (word.upper() in Pos_Words)]
    num = len(pos_words)
    return num

def Neg_words(text):
    words = word_tokenize(text)
    neg_words = [word for word in words if (word.upper() in Neg_Words)]
    num = len(neg_words)
    return num

def Complex_words(text):
    words = word_tokenize(text)
    complex_words = [word for word in words if (word.upper() in Complex_Words)]
    num = len(complex_words)
    return num

def Uncertain_words(text):
    words = word_tokenize(text)
    uncertain_words = [word for word in words if (word.upper() in Uncertain_Words)]
    num = len(uncertain_words)
    return num

In [ ]:
all_data['Positive_RnD_Words'] = all_data['RnD_Text'].apply(Pos_words)

In [ ]:
all_data['Negative_RnD_Words'] = all_data['RnD_Text'].apply(Neg_words)

In [ ]:
all_data['Complex_RnD_Words'] = all_data['RnD_Text'].apply(Complex_words)

In [ ]:
all_data['Uncertain_RnD_Words'] = all_data['RnD_Text'].apply(Uncertain_words)

### Other useful packages to learn:

- `spacy`: https://spacy.io/usage/spacy-101
- `nltk`: https://www.nltk.org/